# 模块七：工业化机器学习工程
**案例背景**：端到端流水线。本 Notebook 涵盖 Pipeline、ColumnTransformer 操作。

# 端到端 Pipeline（Churn 示例）

本笔记本合并了该主题下的多个小实验，按小节依次编排。



---

## 客户流失 Pipeline



来源: 面经知识库 07-end-to-end-pipeline.md §2.1


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(42)
n_samples = 1000

data = {
    "Age": np.random.uniform(18, 70, n_samples),
    "MonthlyCharge": np.random.uniform(20, 120, n_samples),
    "Tenure_Months": np.random.uniform(1, 60, n_samples),
    "Gender": np.random.choice(["Male", "Female"], n_samples),
    "ContractType": np.random.choice(["Month-to-month", "One year", "Two year"], n_samples),
    "PaymentMethod": np.random.choice(["Credit card", "Bank transfer", "Electronic check"], n_samples),
}

df = pd.DataFrame(data)

df.loc[np.random.choice(df.index, 50), "Age"] = np.nan
df.loc[np.random.choice(df.index, 30), "ContractType"] = np.nan

churn_prob = (df["MonthlyCharge"] / 120) * 0.4 + (60 - df["Tenure_Months"]) / 60 * 0.4
churn_prob += np.where(df["ContractType"] == "Month-to-month", 0.2, 0)
df["Churn"] = np.where(churn_prob + np.random.normal(0, 0.1, n_samples) > 0.6, 1, 0)

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ["Age", "MonthlyCharge", "Tenure_Months"]
categorical_features = ["Gender", "ContractType", "PaymentMethod"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [5, 10, None],
    "preprocessor__num__imputer__strategy": ["mean", "median"],
}

print("开始 Pipeline 网格搜索...")
grid_search = GridSearchCV(clf, param_grid, cv=3, scoring="roc_auc", n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"最佳参数组合: {grid_search.best_params_}")
print(f"最佳交叉验证 AUC: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("\n--- 测试集评估结果 ---")
print(classification_report(y_test, y_pred))
print(f"Test ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")